In [33]:
# Import library require for process
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
import warnings as ws
ws.filterwarnings('ignore')




# pca_selection - input -> indep_X and output dep_Y and n_components
# PCA is unsupervised (doesn't use dep_Y), but kept as a parameter for consistency
# with the rest of the pipeline's function signatures.
# Feature scaling MUST happen before PCA, since PCA is variance-based.
def pca_selection(indep_X, dep_Y, n_components):
    sc = StandardScaler()
    X_scaled = sc.fit_transform(indep_X)

    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    print("Explained variance ratio:", pca.explained_variance_ratio_)
    print("Total variance captured:", sum(pca.explained_variance_ratio_))

    pca_df = pd.DataFrame(
        X_pca,
        columns=[f'PC{i+1}' for i in range(n_components)],
        index=indep_X.index
    )
    return pca_df
   
    
#split_scalar - Split the input, output train and test set. then changes the input to scalar value    
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test

# r2_prediction - used for regression method, model prediction evaluate method
def r2_prediction(regressor,X_test,y_test):
     y_pred = regressor.predict(X_test)
     from sklearn.metrics import r2_score
     r2=r2_score(y_test,y_pred)
     return r2
    
# Linear method is used for Linear regression model creation and r2 prediction
def Linear(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.linear_model import LinearRegression
        regressor = LinearRegression()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2   

# SVM_Linear method is used for svm model creation and r2 prediction
def svm_linear(X_train,y_train,X_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'linear')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
# svm_NL method is used for svm_NL model creation and r2 prediction   
def svm_NL(X_train,y_train,X_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'rbf')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     
# Decision method is used for Decision tree model creation and r2 prediction   
def Decision(X_train,y_train,X_test):
        
        # Fitting K-NN to the Training setC
        from sklearn.tree import DecisionTreeRegressor
        regressor = DecisionTreeRegressor(random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     
# random method is used for random forest model creation and r2 prediction  
def random(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.ensemble import RandomForestRegressor
        regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2 
    
# selectk_regression method is used for create dataset with columns name as 'Linear','SVMl','SVMnl','Decision','Random' and index ChiSquare
# and fill the each columns values 
def selectk_regression(acclin,accsvml,accsvmnl,accdes,accrf): 
    
    dataframe=pd.DataFrame(index=['PCA'],columns=['Linear','SVMl','SVMnl','Decision','Random'
                                                                                     ])

    for number,idex in enumerate(dataframe.index):
        
        dataframe['Linear'][idex]=acclin[number]       
        dataframe['SVMl'][idex]=accsvml[number]
        dataframe['SVMnl'][idex]=accsvmnl[number]
        dataframe['Decision'][idex]=accdes[number]
        dataframe['Random'][idex]=accrf[number]
    return dataframe
    

In [43]:
# Read data from file and datatset should without index
dataset=pd.read_csv("prep.csv",index_col=None)
df2=dataset
# Preprocessed by one hot encoding
df2 = pd.get_dummies(df2, drop_first=True)
# assign the input only
indep_X=df2.drop('classification_yes', axis=1)
# assign output only
dep_Y=df2['classification_yes']

# choose the feature selection here using 5 feature 
kbest=pca_selection(indep_X,dep_Y,11)      


Explained variance ratio: [0.26137661 0.07061309 0.06537895 0.05118525 0.04985838 0.04614983
 0.0405221  0.03858053 0.03711253 0.03377813 0.03297591]
Total variance captured: 0.7275313166137041


In [44]:
# Create 5 empty list for each algorithm and split the input and output
# Evalute each algorithmwise r2 score and send selectk_regression funtion
# finally the evalution data represent by table view.
acclin=[]
accsvml=[]
accsvmnl=[]
accdes=[]
accrf=[]

X_train, X_test, y_train, y_test=split_scalar(kbest,dep_Y)  
for i in kbest:  
   
    r2_lin=Linear(X_train,y_train,X_test)
    acclin.append(r2_lin)
    
    r2_sl=svm_linear(X_train,y_train,X_test)    
    accsvml.append(r2_sl)
    
    r2_NL=svm_NL(X_train,y_train,X_test)
    accsvmnl.append(r2_NL)
    
    r2_d=Decision(X_train,y_train,X_test)
    accdes.append(r2_d)
    
    r2_r=random(X_train,y_train,X_test)
    accrf.append(r2_r)
    
    
result=selectk_regression(acclin,accsvml,accsvmnl,accdes,accrf)

In [36]:
result
# 8

,Linear,SVMl,SVMnl,Decision,Random
PCA,0.667804,0.618957,0.914228,1.0,0.990885


In [39]:
result
# 9

,Linear,SVMl,SVMnl,Decision,Random
PCA,0.675469,0.618939,0.911755,0.956597,0.986111


In [42]:
result
# 10

,Linear,SVMl,SVMnl,Decision,Random
PCA,0.671568,0.628273,0.920411,0.956597,0.986111


In [45]:
result
# 11

,Linear,SVMl,SVMnl,Decision,Random
PCA,0.660428,0.631229,0.913208,0.956597,0.993056
